# Performance Optimization

## Objective

Optimize Spark and Delta Lake workloads using common performance and storage optimization techniques.

### Topics Covered

- Cache
- Persist
- Broadcast Join
- Explain Plans
- Delta Time Travel
- Change Data Feed (CDF)
- OPTIMIZE
- VACUUM
- Compute Architecture

### Input Tables

- workspace.default.silver_trips
- workspace.default.silver_weather
- workspace.default.gold_trip_kpis

## Load Tables

In [0]:
from pyspark.sql.functions import broadcast

silver_trips = spark.table(
    "workspace.default.silver_trips"
)

weather = spark.table(
    "workspace.default.silver_weather"
)

gold_trip_kpis = spark.table(
    "workspace.default.gold_trip_kpis"
)

## Cache Example

Caching stores frequently accessed DataFrames in memory to avoid recomputation.

In [0]:
silver_trips.cache()

In [0]:
silver_trips.count()

In [0]:
silver_trips.count()

## Persist Example

Persist provides more control over storage than cache.

In [0]:
from pyspark import StorageLevel

silver_trips.persist(
    StorageLevel.MEMORY_AND_DISK
)

In [0]:
silver_trips.count()

## Explain Plan

Explain displays the Spark execution plan.

In [0]:
silver_trips.explain()

## Broadcast Join

Broadcast joins improve performance when one table is significantly smaller than the other.

silver_trips ≈ millions of rows

silver_weather ≈ a few rows

In [0]:
broadcast_join_df = (
    silver_trips.join(
        broadcast(weather),
        how="cross"
    )
)

In [0]:
broadcast_join_df.explain()

## Delta Table History

Review table versions and operations.

In [0]:
%sql
DESCRIBE HISTORY workspace.default.gold_trip_kpis


## Delta Time Travel

Query a previous version of a Delta table.

In [0]:
%sql
SELECT *
FROM workspace.default.gold_trip_kpis
VERSION AS OF 0

## Enable Change Data Feed

In [0]:
%sql
ALTER TABLE workspace.default.gold_trip_kpis
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
)

## Verify Change Data Feed

In [0]:
%sql
SHOW TBLPROPERTIES workspace.default.gold_trip_kpis

## Read Change Data Feed

In [0]:
%sql
SELECT *
FROM table_changes(
    'workspace.default.gold_trip_kpis',
    3
)

## Optimize Delta Table

OPTIMIZE compacts small files and improves query performance.

In [0]:
%sql
OPTIMIZE workspace.default.gold_trip_kpis

## Vacuum Preview

Identify files eligible for cleanup.

In [0]:
%sql
VACUUM workspace.default.gold_trip_kpis DRY RUN

## Storage Cleanup

Remove obsolete files from Delta storage.

In [0]:
%sql
VACUUM workspace.default.gold_trip_kpis

## Compute Architecture

### Serverless Compute

Advantages:

- No cluster management
- Fast startup
- Automatic scaling
- Reduced operational overhead

### Classic Clusters

Advantages:

- Custom Spark configuration
- Infrastructure control
- Advanced tuning

### Job Clusters

Advantages:

- Created only when needed
- Cost efficient
- Ideal for scheduled workflows

### Current Project

All notebooks, Auto Loader, Lakeflow and Workflows were executed using Serverless Compute.

## Validation

In [0]:
%sql
DESCRIBE HISTORY workspace.default.gold_trip_kpis

In [0]:
%sql
SHOW TBLPROPERTIES workspace.default.gold_trip_kpis


In [0]:
broadcast_join_df.limit(10).display()